In [1]:
from pathlib import Path
import yaml
import uuid

In [2]:
FOLDER = Path("C:/____Moje-MOJE/MyProjects_4Fun/projects/World of Warcraft/rag-pliki/02_chunki")

In [3]:
records = list()

for path in FOLDER.rglob("*.md"):
    text = path.read_text(encoding="utf-8")
    parts = text.split("---", maxsplit=2)

    front_matter = parts[1].strip()
    body = parts[2].strip()

    metadata = yaml.safe_load(front_matter)
    record = {
        "id": metadata.get("chunk_id", "---NO ID"),
        "payload": metadata,
        "embedding_text": f"{metadata.get('chunk_title', '')}\n{body}"
    }

    records.append(record)

In [4]:
from fastembed import TextEmbedding

In [5]:
model = TextEmbedding(model_name="intfloat/multilingual-e5-large")

C:\Users\piotr\AppData\Local\Temp\ipykernel_18032\1956631489.py:1: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  model = TextEmbedding(model_name="intfloat/multilingual-e5-large")


In [6]:
passaged_list = [f"passage: {rec['embedding_text']}" for rec in records]
encoded = list(model.embed(passaged_list))

In [7]:
print(f"LEN RECORDS: {len(records)}")
print(f"PASSAGED LIST: {len(passaged_list)}")
print(f"LEN ENCODED: {len(encoded)}")
print(f"LEN EMBEDDING: {len(encoded[0])}")

LEN RECORDS: 33
PASSAGED LIST: 33
LEN ENCODED: 33
LEN EMBEDDING: 1024


In [22]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [9]:
client = QdrantClient(url="http://localhost:6333")

In [10]:
client.get_collections() # test polaczenia
# rezultat [] = Python połączył się z Qdrantem, ale nie ma jeszcze żadnych kolekcji.

CollectionsResponse(collections=[CollectionDescription(name='wow_lore_chunks')])

In [12]:
COLLECTION_NAME = "wow_lore_chunks"

# client.create_collection(
#     collection_name=COLLECTION_NAME,
#     vectors_config=VectorParams(
#         size=1024,
#         distance=Distance.COSINE,
#     ),
# )

In [19]:
import uuid

In [23]:
points = []

if len(records) == len(encoded):
    for record, embedding in zip(records, encoded):
        chunk_id = record["payload"]["chunk_id"]
        point_id = str(uuid.uuid5(uuid.NAMESPACE_URL, chunk_id)) # stabilne id

        payload = record["payload"].copy()
        payload["embedding_config"] = {
            "embedding_model": "intfloat/multilingual-e5-large",
            "embedding_dim": 1024,
            "embedding_prefix": "passage"
        }
        payload["embedding_text"] = record["embedding_text"]

        point = PointStruct(        # tworzy jeden punkt qdranta, czyli jeden zapis w bazie wektorowej
            id=point_id,
            vector=list(embedding), # wektor chunka, czyli 1024 liczby z modelu E5
            payload=payload         # metadane i tekst chunka, np. chunk_title, entity_name, embedding_text
        )                           # to taki punkt w 1024 wymiarach; 1024 liczby wspólnie wyznaczają jeden punkt. Nie jako     przecięcie boków bryły, bardziej jako adres/współrzędne w bardzo wielowymiarowej mapie znaczeń.

        points.append(point)

client.upsert(# Wstaw punkt, a jeśli punkt o tym ID już istnieje, nadpisz go.
    collection_name=COLLECTION_NAME,
    points=points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [25]:
client.count(collection_name=COLLECTION_NAME) # check czy dziala

CountResult(count=33)

In [74]:
query = "Orweyna visions goddess call Radiant Song outsiders Haranir mission"

In [75]:
embedded_query = list(model.embed([f"query: {query}"]))
query_vector = list(embedded_query[0])

In [76]:
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
    with_payload=True,
)

In [77]:
for index, point in enumerate(results.points, start=1):
    payload = point.payload
    text_preview = payload["embedding_text"][:300].replace("\n", " ")

    print(f"{index}. score: {point.score:.3f}")
    print(f"title: {payload['chunk_title']}")
    print(f"entity: {payload['entity_name']} ({payload['entity_type']})")
    print(f"topic: {payload['topic']}")
    print(f"text: {text_preview}...")
    print("-" * 80)

1. score: 0.864
title: Orweyna — overview and role
entity: Orweyna (character)
topic: identity_role_radiant_song
text: Orweyna — overview and role # Orweyna — overview and role  ## Contextual header This chunk summarizes Orweyna as a character: her role among the haranir, her work against Black Blood, her willingness to work with outsiders, and her connection to the Radiant Song.  ## Chunk text Orweyna is the leader...
--------------------------------------------------------------------------------
2. score: 0.847
title: Orweyna — visions from Azeroth and the Ringing Deeps
entity: Orweyna (character)
topic: azeroth_visions_goddess_call_ringing_deeps
text: Orweyna — visions from Azeroth and the Ringing Deeps # Orweyna — visions from Azeroth and the Ringing Deeps  ## Contextual header This chunk focuses on Orweyna’s visions, her goddess, and her tendency to follow the call of Azeroth even when it isolates her from her people.  ## Chunk text With the ot...
--------------------------------